<a href="https://colab.research.google.com/github/Anderfg13/traductor-jerga-dialectal-slm/blob/develop/finetuning/fase3_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline completo de la Fase 3 — Google Colab

Corre, en orden y sobre GPU gratuita de Colab (T4/L4):

1. **Entrenar** 4 adaptadores LoRA: Generador 1 (Groq), 2 (Cohere), 3 (Google) y la **mezcla** de los tres.
2. **Fusionar** los adaptadores de los 3 generadores con TIES, DARE+TIES y promedio lineal.
3. **Predecir**: traducir el test común (`evaluation/test_comun.json`) con 8 modelos (base sin ajustar, 4 adaptadores, 3 fusiones).
4. **Medir** BLEU y chrF de cada uno y armar la tabla comparativa.
5. **Descargar** un `.zip` con todo para traerlo al repo local.

**Antes de empezar**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**. Las celdas se corren en orden; si Colab se desconecta, vuelve a correr desde la celda 1 (las partes ya hechas se saltan solas si sus carpetas existen).

Tiempo estimado en T4: ~15-30 min por adaptador (el Generador 1 con 189 ejemplos tardó ~9 min; ahora son ~475), ~5 min por fusión, ~5 min por modelo en predicciones: **2 a 3 horas** en total.

In [1]:
# 1. Confirmar que hay GPU asignada
!nvidia-smi

Thu Oct  8 03:56:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Clonar el repo (público) con rutas absolutas y solo si hace falta;
# si ya existe, traer lo último de la rama develop.
import os
%cd /content
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone -b develop https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm
!git pull -q origin develop

/content
Cloning into 'traductor-jerga-dialectal-slm'...
remote: Enumerating objects: 830, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 830 (delta 7), reused 9 (delta 3), pack-reused 783 (from 1)
Receiving objects: 100% (830/830), 68.77 MiB | 16.08 MiB/s, done.
Resolving deltas: 100% (592/592), done.
Updating files: 100% (479/479), done.
/content/traductor-jerga-dialectal-slm


In [3]:
# 3. Dependencias puntuales (NO requirements.txt completo: Colab ya trae
# torch con CUDA). Se desinstala torchao porque la versión vieja
# preinstalada rompe peft (ver CONTEXTO_PROYECTO.md, "CÓMPUTO PESADO").
!pip install -q transformers peft accelerate python-dotenv sacrebleu
!pip uninstall -y -q torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.9 MB/s eta 0:00:00


In [4]:
# Cronómetro: cada paso pesado guarda cuánto tardó en tiempos_fase3.json (va dentro del zip final),
# porque Colab no conserva los tiempos de las celdas si se reinicia el entorno.
import json, os, time
TIEMPOS_PATH = "tiempos_fase3.json"
def registrar(nombre, t0):
    t = json.load(open(TIEMPOS_PATH)) if os.path.exists(TIEMPOS_PATH) else {}
    t[nombre] = round(time.time() - t0, 1)
    json.dump(t, open(TIEMPOS_PATH, "w"), indent=1)
    print(f"[tiempo] {nombre}: {t[nombre] / 60:.1f} min")
import subprocess
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
except Exception:
    gpu = "sin GPU"
t = json.load(open(TIEMPOS_PATH)) if os.path.exists(TIEMPOS_PATH) else {}
t["gpu"] = gpu
json.dump(t, open(TIEMPOS_PATH, "w"), indent=1)
print("GPU:", gpu)

GPU: Tesla T4


## Parte A — Datos

In [5]:
# 4. Armar el dataset de la mezcla (1+2+3) y el test común de evaluación.
!python generation/mezclar_datasets.py
!python evaluation/construir_test_comun.py

train: 1496 variantes, 81 semillas
val: 183 variantes, 10 semillas
test: 165 variantes, 9 semillas
Verificación OK: ninguna semilla repetida entre splits. Guardado en /content/traductor-jerga-dialectal-slm/generation/splits/dataset_mezcla
174 entradas en test_comun.json: {'oro': 9, 'generador1': 54, 'generador2': 55, 'generador3': 56}


## Parte B — Entrenar los 4 adaptadores

In [6]:
# Entrenar: Generador 1 (Groq).
# IMPORTANTE: el repo trae un adaptador VIEJO de este generador (entrenado en la Sesión 19 con
# 189 ejemplos de otro reparto de semillas). Se borra para FORZAR el reentrenamiento con el
# reparto nuevo; si no, esta celda lo saltaría y la comparación con los demás sería inválida
# (además 2 semillas de test de hoy estaban en su entrenamiento). Solo borra la copia de Colab.
import shutil
shutil.rmtree("finetuning/checkpoints/generador1", ignore_errors=True)
_t0 = time.time()
!python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_generador1 --todos
registrar("entrenar_generador1", _t0)
!python -c "import json; d=json.load(open('finetuning/checkpoints/generador1/loss_log.json')); n=len(d['train'])/len(d['eval']); print('ejemplos de entrenamiento implícitos:', round(n), '(debe ser 475, no 189)'); assert round(n)==475"

Cargando Qwen/Qwen2.5-3B-Instruct en bfloat16 (sin cuantizar) para entrenar, dispositivo: GPU (CUDA)...
config.json: 100% 661/661 [00:00<00:00, 2.68MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 18.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 79.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 95.2MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 118MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 75.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/2.20G [00:00<?, ?B/s]         

Reconstructing (incomplete total...):  78% 4.82G/6.17G [00:31<00:08, 164MB/s,  201MB/s  ]
Reconstructing (incomplete total...):  83% 5.09G/6.17G [00:32<00:06, 160MB/s,  164MB/s  ]
Reconstructing (incomplete total...):  95% 5.85G/6.17G [00:47<00:04, 71.2MB/s,  105MB/s  ]
Reconstru

In [7]:
# Entrenar: Generador 2 (Cohere). Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/generador2/adapter"):
    print("Ya existe finetuning/checkpoints/generador2/adapter, se salta.")
else:
    _t0 = time.time()
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_generador2 --todos
    registrar("entrenar_generador2", _t0)

Ya existe finetuning/checkpoints/generador2/adapter, se salta.


In [8]:
# Entrenar: Generador 3 (Google). Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/generador3/adapter"):
    print("Ya existe finetuning/checkpoints/generador3/adapter, se salta.")
else:
    _t0 = time.time()
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_generador3 --todos
    registrar("entrenar_generador3", _t0)

Ya existe finetuning/checkpoints/generador3/adapter, se salta.


In [9]:
# Entrenar: Mezcla 1+2+3. Entrenamiento completo con validación y
# selección del mejor checkpoint (EarlyStopping). Se salta si ya existe.
import os
if os.path.isdir("finetuning/checkpoints/mezcla/adapter"):
    print("Ya existe finetuning/checkpoints/mezcla/adapter, se salta.")
else:
    _t0 = time.time()
    !python finetuning/entrenar_lora.py --dataset-dir generation/splits/dataset_mezcla --todos
    registrar("entrenar_mezcla", _t0)

Ya existe finetuning/checkpoints/mezcla/adapter, se salta.


In [10]:
# Curvas de pérdida de los 4 entrenamientos (para el paper y para revisar sobreajuste)
import json
for c in ["generador1", "generador2", "generador3", "mezcla"]:
    print("=====", c)
    print(open(f"finetuning/checkpoints/{c}/loss_log.json").read()[:1500])

===== generador1
{
  "train": [
    {
      "paso": 1,
      "epoca": 0.002,
      "loss": 4.636298179626465
    },
    {
      "paso": 2,
      "epoca": 0.004,
      "loss": 2.1903159618377686
    },
    {
      "paso": 3,
      "epoca": 0.006,
      "loss": 2.6992273330688477
    },
    {
      "paso": 4,
      "epoca": 0.008,
      "loss": 2.591304063796997
    },
    {
      "paso": 5,
      "epoca": 0.011,
      "loss": 2.771256923675537
    },
    {
      "paso": 6,
      "epoca": 0.013,
      "loss": 2.0356333255767822
    },
    {
      "paso": 7,
      "epoca": 0.015,
      "loss": 1.0013580322265625
    },
    {
      "paso": 8,
      "epoca": 0.017,
      "loss": 0.7229992151260376
    },
    {
      "paso": 9,
      "epoca": 0.019,
      "loss": 1.2176483869552612
    },
    {
      "paso": 10,
      "epoca": 0.021,
      "loss": 1.0868659019470215
    },
    {
      "paso": 11,
      "epoca": 0.023,
      "loss": 1.1071066856384277
    },
    {
      "paso": 12,
      "epo

In [11]:
# Verificar que los 4 adaptadores salieron con la MISMA configuración de LoRA
# (condición para que la comparación de PI1 sea válida). Debe terminar en OK.
!python finetuning/verificar_config_identica.py

== Configuración en el código (entrenar_lora.py), idéntica para todo dataset ==
LoraConfig: {'r': '8', 'lora_alpha': '16', 'lora_dropout': '0.05', 'bias': "'none'", 'target_modules': "['q_proj', 'k_proj', 'v_proj', 'o_proj']"}
TrainingArguments: {'per_device_train_batch_size': '1', 'learning_rate': '0.0002', 'num_train_epochs': 'epocas'}
Constantes: {'EPOCAS_COMPLETO_MAX': '10', 'PATIENCE_DEFAULT': '2', 'SEMILLA_ALEATORIA': '42', 'MAX_LENGTH': '512'}

== adapter_config.json guardados ==
  generador1: r=8 alpha=16 dropout=0.05 bias=none módulos=['k_proj', 'o_proj', 'q_proj', 'v_proj']
  generador2: r=8 alpha=16 dropout=0.05 bias=none módulos=['k_proj', 'o_proj', 'q_proj', 'v_proj']
  generador3: r=8 alpha=16 dropout=0.05 bias=none módulos=['k_proj', 'o_proj', 'q_proj', 'v_proj']
  mezcla: r=8 alpha=16 dropout=0.05 bias=none módulos=['k_proj', 'o_proj', 'q_proj', 'v_proj']

OK: 4 adaptador(es) con configuración idéntica entre sí y con el código; 0 pendiente(s) de comparar.


## Parte C — Fusión de adaptadores (PI2)

TIES, DARE+TIES y promedio lineal de los adaptadores de los generadores 1, 2 y 3. `--densidad 0.5`: se conserva la mitad de los parámetros de cada adaptador (valor estándar; se puede variar).

In [12]:
G = "finetuning/checkpoints/generador{}/adapter"
ADAPTADORES = f"{G.format(1)} {G.format(2)} {G.format(3)}"
for metodo in ["ties", "dare_ties", "linear"]:
    salida = f"finetuning/checkpoints/fusion_{metodo}"
    if os.path.isdir(salida + "/fusion"):
        print(f"Ya existe {salida}/fusion, se salta.")
    else:
        _t0 = time.time()
        !python merging/fusionar_adaptadores.py --metodo {metodo} --adaptadores {ADAPTADORES} --salida {salida}
        registrar(f"fusion_{metodo}", _t0)

Loading weights: 100% 434/434 [00:01<00:00, 224.76it/s]
Fusión 'ties' de 3 adaptadores guardada en finetuning/checkpoints/fusion_ties/fusion
[tiempo] fusion_ties: 0.3 min
Loading weights: 100% 434/434 [00:01<00:00, 224.15it/s]
Fusión 'dare_ties' de 3 adaptadores guardada en finetuning/checkpoints/fusion_dare_ties/fusion
[tiempo] fusion_dare_ties: 0.3 min
Loading weights: 100% 434/434 [00:01<00:00, 233.79it/s]
Fusión 'linear' de 3 adaptadores guardada en finetuning/checkpoints/fusion_linear/fusion
[tiempo] fusion_linear: 0.2 min


### Fusión guiada por destilación multi-maestro

Un adaptador estudiante (arranca desde la fusión TIES) aprende a imitar el promedio de los 3 adaptadores-maestro sobre el dataset mezcla. Ver `merging/destilacion_multimaestro.py`. Tarda ~20-40 min en T4.
    registrar("destilacion", _t0)

In [ ]:
if os.path.isdir("finetuning/checkpoints/destilacion/estudiante"):
    print("Ya existe la destilación, se salta.")
else:
    _t0 = time.time()
    !python merging/destilacion_multimaestro.py --maestros finetuning/checkpoints/generador1/adapter finetuning/checkpoints/generador2/adapter finetuning/checkpoints/generador3/adapter --inicial finetuning/checkpoints/fusion_ties/fusion --dataset-dir generation/splits/dataset_mezcla --salida finetuning/checkpoints/destilacion
    registrar("destilacion", _t0)

Loading weights: 100% 434/434 [00:01<00:00, 244.73it/s]
Maestros: 3 | train: 1496 | val: 183 | GPU: True
Parámetros entrenables (solo el estudiante): 3,686,400


## Parte D — Predicciones sobre el test común

In [ ]:
MODELOS = {
    "baseline": None,
    "generador1": "finetuning/checkpoints/generador1/adapter",
    "generador2": "finetuning/checkpoints/generador2/adapter",
    "generador3": "finetuning/checkpoints/generador3/adapter",
    "mezcla": "finetuning/checkpoints/mezcla/adapter",
    "fusion_ties": "finetuning/checkpoints/fusion_ties/fusion",
    "fusion_dare_ties": "finetuning/checkpoints/fusion_dare_ties/fusion",
    "fusion_linear": "finetuning/checkpoints/fusion_linear/fusion",
    "destilacion": "finetuning/checkpoints/destilacion/estudiante",
}
for nombre, adapter in MODELOS.items():
    if os.path.exists(f"evaluation/predicciones/{nombre}.json"):
        print(f"Ya existen las predicciones de {nombre}, se salta.")
        continue
    flag = f"--adapter {adapter}" if adapter else ""
    _t0 = time.time()
    !python evaluation/generar_predicciones.py --nombre {nombre} {flag}
    registrar(f"predicciones_{nombre}", _t0)

## Parte E — Métricas y tabla comparativa

In [ ]:
os.makedirs("evaluation/reportes", exist_ok=True)
for nombre in MODELOS:
    print("=====", nombre)
    !python evaluation/metricas_automaticas.py --predicciones evaluation/predicciones/{nombre}.json --referencias evaluation/test_comun.json --campo-prediccion traduccion_modelo --salida evaluation/reportes/{nombre}.md --salida-json evaluation/reportes/{nombre}.json | head -12
!python evaluation/tabla_comparativa.py

## Parte F — Empaquetar y descargar

In [ ]:
# Solo lo necesario: adaptadores finales (NO los checkpoints intermedios del
# Trainer, que pesan mucho), pérdidas, predicciones y reportes.
!rm -f resultados_fase3.zip
!zip -rq resultados_fase3.zip finetuning/checkpoints/generador1/adapter finetuning/checkpoints/generador1/loss_log.json finetuning/checkpoints/generador2/adapter finetuning/checkpoints/generador2/loss_log.json finetuning/checkpoints/generador3/adapter finetuning/checkpoints/generador3/loss_log.json finetuning/checkpoints/mezcla/adapter finetuning/checkpoints/mezcla/loss_log.json finetuning/checkpoints/fusion_ties/fusion finetuning/checkpoints/fusion_dare_ties/fusion finetuning/checkpoints/fusion_linear/fusion finetuning/checkpoints/destilacion/estudiante evaluation/predicciones evaluation/reportes evaluation/comparacion_fase3.md tiempos_fase3.json
!ls -lh resultados_fase3.zip
from google.colab import files
files.download("resultados_fase3.zip")